# تقييم سَنَد — Correctness · Faithfulness · Citation accuracy

**طريقة الاستخدام:**
1. ارفعي ملفات المقيّمين (`grader_1_grading.csv` و`grader_2_grading.csv`) في الخلية الثانية.
2. (اختياري) ارفعي `final.csv` بعد حسم العضو الثالث، يحتوي: `question_id`, `final_correctness_0_2`, `final_faithful_0_1`, `final_citations_correct_0_1`, `answerable`, `gold_in_top5`.
3. شغّلي الخلايا بالترتيب.

النتائج تُحفظ في ملفات CSV تنزلينها في الخلية الأخيرة.

In [ ]:
# الخلية 1: التثبيت (مرة واحدة)
!pip -q install pandas scikit-learn

In [ ]:
# الخلية 2: رفع الملفات
from google.colab import files
uploaded = files.upload()
print("الملفات المرفوعة:", list(uploaded.keys()))

In [ ]:
# الخلية 3: الإعدادات
import os
G1 = "grader_1_grading.csv"
G2 = next((f for f in uploaded if f.startswith("grader_2")), "grader_2_grading.csv")
FINAL = next((f for f in uploaded if f.startswith("final")), None)   # اختياري
print("المقيّم 1:", G1, "| المقيّم 2:", G2, "| النهائي:", FINAL or "غير موجود (النتائج مؤقتة)")

In [ ]:
# الخلية 4: قراءة الملفات (تتعامل مع الفاصلة المنقوطة والترميز العربي)
import pandas as pd
from sklearn.metrics import cohen_kappa_score

def load(path):
    for enc in ("utf-8-sig", "cp1256", "utf-8"):
        for sep in (";", ","):
            try:
                d = pd.read_csv(path, encoding=enc, sep=sep, engine="python")
                if "question_id" in d.columns and d.shape[1] >= 8:
                    return d
            except Exception:
                pass
    raise ValueError(f"تعذّرت قراءة {path}")

a = load(G1)
b = load(G2)
assert (a["question_id"].values == b["question_id"].values).all(), "ترتيب الأسئلة مختلف بين الملفين"
ids = a["question_id"]
n = len(ids)
print("عدد الأسئلة:", n)

# الامتناع: الحقل من النظام
abst = a["abstained"].astype(str).str.upper().eq("TRUE")

# عدد الاستشهادات لكل سؤال من عمود cited_articles
def n_cits(s):
    s = str(s).strip()
    return 0 if s in ("", "—", "nan") else len([x for x in s.split(";") if x.strip()])
ncit = a["cited_articles"].apply(n_cits)

In [ ]:
# الخلية 5: اكتمال التعبئة (تحقّقي منه قبل أي رقم)
cols = {"Correctness": ("grader_correctness_0_2",), "Faithfulness": ("grader_faithful_0_1",),
        "Citations": ("grader_citations_correct_0_1",)}
print(f"{'المعيار':<14}{'المقيّم 1':>12}{'المقيّم 2':>12}")
for name, (c,) in cols.items():
    print(f"{name:<14}{a[c].notna().sum():>9}/{n}{b[c].notna().sum():>9}/{n}")
ref_empty = a["reference_answer"].isna().sum() if "reference_answer" in a else n
print(f"\nreference_answer فاضي في {ref_empty}/{n} سؤال")
if ref_empty:
    print("⚠️ لا تعتمدين الأرقام في التقرير قبل كتابة reference_answer.")

In [ ]:
# الخلية 6: الملف النهائي (إذا وُجد)
final = None
if FINAL:
    f = load(FINAL).set_index("question_id").reindex(ids)
    final = f
    print("نهائي: نعم | الأعمدة:", [c for c in f.columns if c.startswith("final_") or c in ("answerable","gold_in_top5")])
else:
    print("نهائي: لا، كل النتائج مؤقتة")

In [ ]:
# الخلية 7: Correctness (0/1/2)
score_c, status_c = ((final["final_correctness_0_2"].astype(float), "نهائية")
                     if final is not None and final["final_correctness_0_2"].notna().all()
                     else ((a["grader_correctness_0_2"].astype(float) + b["grader_correctness_0_2"].astype(float)) / 2,
                           "مؤقتة (متوسط المقيّمين)"))
print("== Correctness ==", status_c)
print(f"المتوسط: {score_c.mean():.2f} من 2 → {score_c.mean()/2:.1%}")
print(f"إجابات كاملة (2): {int((score_c==2).sum())}/{n} = {(score_c==2).mean():.1%}")
print(f"جزئية (1):        {int((score_c==1).sum())}/{n}")
print(f"خاطئة (0):        {int((score_c==0).sum())}/{n} = {(score_c==0).mean():.1%}")

In [ ]:
# الخلية 8: Faithfulness (على الأجوبة غير الممتنعة فقط)
mask = ~abst
if final is not None and final["final_faithful_0_1"].notna().all():
    faith = final["final_faithful_0_1"].astype(float)[mask]; status_f = "نهائية"
else:
    both = a["grader_faithful_0_1"].notna() & b["grader_faithful_0_1"].notna() & mask
    faith = ((a.loc[both,"grader_faithful_0_1"].astype(float) + b.loc[both,"grader_faithful_0_1"].astype(float)) / 2)
    status_f = f"مؤقتة (على {int(both.sum())} سؤال درجه المقيّمان)"
print("== Faithfulness ==", status_f)
if len(faith):
    print(f"نسبة الأجوبة المدعومة: {faith.mean():.1%}  (من {len(faith)} جواب غير ممتنع)")
else:
    print("لا توجد أجوبة مدعومة بالدرجات بعد. أكملي عمود grader_faithful_0_1 لكل المقيّمين.")

In [ ]:
# الخلية 9: Citation accuracy (وزن كل استشهاد)
if final is not None and final["final_citations_correct_0_1"].notna().all():
    cit = final["final_citations_correct_0_1"].astype(float); status_c2 = "نهائية"
else:
    both = a["grader_citations_correct_0_1"].notna() & b["grader_citations_correct_0_1"].notna()
    cit = ((a["grader_citations_correct_0_1"].astype(float) + b["grader_citations_correct_0_1"].astype(float)) / 2)[both]
    status_c2 = f"مؤقتة (على {int(both.sum())} سؤال درجه المقيّمان)"
w = ncit.reindex(cit.index).astype(float)
print("== Citation accuracy ==", status_c2)
if w.sum() > 0:
    print(f"دقة الاستشهادات (موزونة بعدد الاستشهادات): {(cit*w).sum()/w.sum():.1%}  ({int(w.sum())} استشهاد)")
else:
    print("لا توجد استشهادات مُقيّمة بعد.")

In [ ]:
# الخلية 10: الاتفاق بين المقيّمين (kappa)
def agreement(col, weights=None):
    both = a[col].notna() & b[col].notna()
    x = a.loc[both, col].astype(int); y = b.loc[both, col].astype(int)
    if len(x) == 0:
        return "لا يوجد سؤال مشترك بعد"
    out = f"اتفاق {(x==y).mean():.1%}  |  الاختلافات {(x!=y).sum()}"
    if len(set(x) | set(y)) > 1:
        out += f"  |  kappa {cohen_kappa_score(x, y, weights=weights):.3f}"
    return out

print("Correctness:  ", agreement("grader_correctness_0_2", "linear"))
print("Faithfulness: ", agreement("grader_faithful_0_1"))
print("Citations:    ", agreement("grader_citations_correct_0_1"))

In [ ]:
# الخلية 11: الامتناع (يحتاج عمود answerable في final.csv)
if final is not None and "answerable" in final and final["answerable"].notna().all():
    ans = final["answerable"].astype(str).str.upper().eq("TRUE")
    ok_abst = ((~ans) & abst).sum(); bad_abst = (ans & abst).sum()
    print(f"امتناع صحيح على غير القابلة: {ok_abst}/{(~ans).sum()}")
    print(f"امتناع خاطئ على القابلة:     {bad_abst}/{ans.sum()}")
else:
    print("دقة الامتناع تحتاج عمود answerable في final.csv")

In [ ]:
# الخلية 12: الاختلافات للحسم + ملف الملخص
dis = a.loc[a["grader_correctness_0_2"].astype(float) != b["grader_correctness_0_2"].astype(float),
            ["question_id", "question"]].copy()
dis["grader_1"] = a.loc[dis.index, "grader_correctness_0_2"]
dis["grader_2"] = b.loc[dis.index, "grader_correctness_0_2"]
dis.to_csv("correctness_disagreements.csv", index=False, encoding="utf-8-sig")

summary = pd.DataFrame([
    ("Correctness (mean/2)", round(score_c.mean()/2, 4), status_c),
    ("Full-correct share", round((score_c==2).mean(), 4), status_c),
    ("Faithfulness", round(faith.mean(), 4) if len(faith) else None, status_f),
    ("Citation accuracy (weighted)", round((cit*w).sum()/w.sum(), 4) if w.sum() else None, status_c2),
], columns=["metric", "value", "status"])
summary.to_csv("sanad_eval_summary.csv", index=False, encoding="utf-8-sig")
print(summary.to_string(index=False))

from google.colab import files as _f
_f.download("sanad_eval_summary.csv")
_f.download("correctness_disagreements.csv")